# Main Figure 2 - Neonatal mouse pup Xenium

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `sfplot-manuscript/figures/fig. 1.ipynb`, `CellGPS manuscript/figures/meta-structures.R`, and mouse pup t-by-c output tables.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Figure 2a-f, shared setup. Imports plotting and table utilities used by the mouse pup Xenium panels.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Figure 2a-b. Loads the neonatal mouse pup Xenium cells, builds the x/y/celltype table, and computes the whole-tissue Cell-GPS StructureMap heatmap.
#

from sfplot import load_xenium_data, compute_cophenetic_distances_from_df, plot_cophenetic_heatmap, circle_heatmap

XENIUM_DIR = Path("Y:/long/10X_datasets/Xenium/Xenium_5K/Xenium_Prime_Mouse_Pup_FFPE_outs")
TBC_DIR = Path("Y:/long/10X_datasets/Xenium/Xenium_5K/t_by_c_result/t_by_c_Xenium_Prime_Mouse_Pup_FFPE_outs")
OUTPUT_DIR = Path("outputs/main_figure_2_mouse_pup")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load cell centroids and annotations.
adata = load_xenium_data(str(XENIUM_DIR), normalize=False)
adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]
cluster_key = "annotation" if "annotation" in adata.obs else "Cluster"
points = adata.obs[["x", "y", cluster_key]].rename(columns={cluster_key: "celltype"})

# Compute and save the whole-tissue mouse pup StructureMap.
row_coph, col_coph = compute_cophenetic_distances_from_df(points, x_col="x", y_col="y", celltype_col="celltype")
row_coph.to_csv(OUTPUT_DIR / "StructureMap_table_Mouse_Pup_FFPE.csv")
plot_cophenetic_heatmap(row_coph, matrix_name="row_coph", output_filename=str(OUTPUT_DIR / "StructureMap_of_Mouse_Pup_FFPE.pdf"), sample="Mouse_Pup_FFPE")


In [ ]:
# Panel mapping:
# - Figure 2c-e. Generates the selected biological spatial panels for retina-related, fibroblast, and keratinocyte/sebaceous cluster groups.
#

# Generate selected biological spatial panels used in Figure 2.
def save_cluster_panel(cluster_ids, title, filename):
    subset = adata[adata.obs["Cluster"].astype(int).isin(cluster_ids)].copy()
    coords = subset.obsm["spatial"]
    labels = subset.obs["Cluster"].astype(str).to_numpy()
    fig, ax = plt.subplots(figsize=(7, 10))
    for label in sorted(set(labels), key=lambda x: int(x)):
        mask = labels == label
        ax.scatter(coords[mask, 0], coords[mask, 1], s=1.0, alpha=0.7, label=label, rasterized=True, linewidths=0)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.legend(title="Cluster", frameon=False, markerscale=4)
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.close(fig)

save_cluster_panel([30, 33, 37], "Retina-related clusters", "mouse_pup_retina_clusters.pdf")
save_cluster_panel([1, 5], "Fibroblast subtype spatial pattern", "mouse_pup_fibroblast_clusters.pdf")
save_cluster_panel([3, 42], "Keratinocyte/sebaceous spatial pattern", "mouse_pup_keratinocyte_clusters.pdf")


In [ ]:
# Panel mapping:
# - Figure 2f. Recreates the transcript-by-cell circular heatmap for the selected marker genes used in the mouse pup figure.
#

# Transcript-by-cell circle heatmap for representative genes.
result = pd.read_csv(TBC_DIR / "t_and_c_result_Xenium_Prime_Mouse_Pup_FFPE_outs.csv", index_col=0)
transcript_pct = pd.read_csv(TBC_DIR / "transcript_table_percentage_Xenium_Prime_Mouse_Pup_FFPE_outs.csv", index_col=0)
genes = ["Gsdma3", "Kcna10", "Ly6k", "Dsg1a", "Pou4f3"]
cell_types = [c for c in transcript_pct.columns if pd.notna(c) and str(c).lower() != "nan"]
fig, ax, legend_axes = circle_heatmap(result.reindex(genes, columns=cell_types), transcript_pct.reindex(genes, columns=cell_types))
fig.savefig(OUTPUT_DIR / "TCSplot_Mouse_Pup_selected_genes.pdf", bbox_inches="tight")
plt.close(fig)
